# homeassistant / airquality / pm25

**Intent:** Output a corrected PM2.5 for a room's air purifier from its raw reading and the room temperature, removing the part of the reading that follows temperature and levels off, so readings are comparable across rooms. The signature is not yet defined: the scaffold names are placeholders (`homeassistant_airquality_in` and `homeassistant_airquality_out`); humidity and fan speed join the inputs if shown to matter.

**Notes:**
- Purpose: Make the four purifiers' PM2.5 readings comparable, so a reading means the same particles in every room
- Data splits: `training` the dining and lounge purifier PM2.5 readings with room temperature, humidity and fan speed, in time order; `validation` and `backtest` later periods held back for acceptance; `golden` the fixed readings and expected corrected values that become `golden.json`. Nothing is loaded yet

## Setup

- **Stage:** none, preparation for the stages below.
- **Intent:** load `env` and declare the model name, its targets and the dataset names as constants (`model.md` 6.6).

In [ ]:
from workbench.data import golden_data
from workbench.env import init_env
from workbench.project import load_signature, scratch_project_dir, signature_names

env = init_env()

WORKSPACE = "homeassistant/airquality/pm25"
MODEL_NAME = "homeassistant_airquality_pm25"
TARGETS = [{"kind": "served"}]
signature = load_signature(WORKSPACE)
INPUTS, OUTPUTS = signature_names(signature)
GOLDEN = "airquality_pm25_golden"
CONSTANTS = {"homeassistant_airquality_out": 1}

## Capture Data

- **Stage:** 1, capture data (`model.md` 7.1).
- **Intent:** load the `training` and `validation` data.

## Fit

- **Stage:** 3, fit.
- **Intent:** fit the model on `training` only (R6).

## Evaluate

- **Stage:** 4, evaluate.
- **Intent:** check the fit on `validation` and `backtest` through the shipped runtime and run its acceptance check with `evaluate_acceptance` (R4, R14).

## Package

- **Stage:** 5, package.
- **Intent:** write the golden samples, build the signature and write the project with its evaluation (R1, R3, R14).

In [ ]:
golden_pl, golden_path = golden_data(WORKSPACE, GOLDEN, lambda: {"homeassistant_airquality_in": [1.0, None], "expect": ["ok", "error"]})

project_dir = scratch_project_dir(WORKSPACE)
print("no model is fitted yet, so nothing is packaged")

## Register

- **Stage:** 6, register.
- **Intent:** register the verified project with the alias `preview` when it has a served target. `fab systest` runs this section and `fab unittest` does not (`model.md` 11.5).

In [ ]:
from workbench.model import register_preview

version = register_preview(project_dir, MODEL_NAME, env["MLFLOW_TRACKING_URI"], TARGETS)
print("registered version", version)